# LeNet Lab Solution


## Load Data

Load the MNIST data via `tf.keras.datasets.mnist`.

You do not need to modify this section.

In [ ]:
import tensorflow as tf
import numpy as np

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# Reshape to (N, 28, 28, 1) and normalize to [0,1] float32
X_train = X_train[..., np.newaxis].astype(np.float32) / 255.0
X_test  = X_test[..., np.newaxis].astype(np.float32) / 255.0
y_train = y_train.astype(np.int32)
y_test  = y_test.astype(np.int32)

# Split off validation set (last 5000 of training)
X_validation, y_validation = X_train[-5000:], y_train[-5000:]
X_train, y_train           = X_train[:-5000], y_train[:-5000]

assert(len(X_train) == len(y_train))
assert(len(X_validation) == len(y_validation))
assert(len(X_test) == len(y_test))

print()
print("Image Shape: {}".format(X_train[0].shape))
print()
print("Training Set:   {} samples".format(len(X_train)))
print("Validation Set: {} samples".format(len(X_validation)))
print("Test Set:       {} samples".format(len(X_test)))

In [2]:
import numpy as np

# Pad images with 0s
X_train      = np.pad(X_train, ((0,0),(2,2),(2,2),(0,0)), 'constant')
X_validation = np.pad(X_validation, ((0,0),(2,2),(2,2),(0,0)), 'constant')
X_test       = np.pad(X_test, ((0,0),(2,2),(2,2),(0,0)), 'constant')
    
print("Updated Image Shape: {}".format(X_train[0].shape))

Updated Image Shape: (32, 32, 1)


In [9]:
from sklearn.utils import shuffle

X_train, y_train = shuffle(X_train, y_train)

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

## SOLUTION: Implement LeNet-5
Implement the [LeNet-5](http://yann.lecun.com/exdb/lenet/) neural network architecture.

This is the only cell you need to edit.
### Input
The LeNet architecture accepts a 32x32xC image as input, where C is the number of color channels. Since MNIST images are grayscale, C is 1 in this case.

### Architecture
**Layer 1: Convolutional.** The output shape should be 28x28x6.

**Activation.** Your choice of activation function.

**Pooling.** The output shape should be 14x14x6.

**Layer 2: Convolutional.** The output shape should be 10x10x16.

**Activation.** Your choice of activation function.

**Pooling.** The output shape should be 5x5x16.

**Flatten.** Flatten the output shape of the final pooling layer such that it's 1D instead of 3D. The easiest way to do is by using `tf.contrib.layers.flatten`, which is already imported for you.

**Layer 3: Fully Connected.** This should have 120 outputs.

**Activation.** Your choice of activation function.

**Layer 4: Fully Connected.** This should have 84 outputs.

**Activation.** Your choice of activation function.

**Layer 5: Fully Connected (Logits).** This should have 10 outputs.

### Output
Return the result of the 2nd fully connected layer.

In [ ]:
class LeNet(tf.keras.Model):
    def __init__(self):
        super().__init__()
        init = tf.keras.initializers.TruncatedNormal(mean=0, stddev=0.1)
        zero = tf.keras.initializers.Zeros()

        # Layer 1: Convolutional. Input = 32x32x1. Output = 28x28x6.
        self.conv1 = tf.keras.layers.Conv2D(6, (5, 5), padding='valid',
                         kernel_initializer=init, bias_initializer=zero, name='conv1')
        self.pool1 = tf.keras.layers.MaxPool2D((2, 2), (2, 2), padding='valid')

        # Layer 2: Convolutional. Output = 10x10x16.
        self.conv2 = tf.keras.layers.Conv2D(16, (5, 5), padding='valid',
                         kernel_initializer=init, bias_initializer=zero, name='conv2')
        self.pool2 = tf.keras.layers.MaxPool2D((2, 2), (2, 2), padding='valid')

        # Flatten. Input = 5x5x16. Output = 400.
        self.flat = tf.keras.layers.Flatten()

        # Layer 3: Fully Connected. Input = 400. Output = 120.
        self.fc1 = tf.keras.layers.Dense(120, kernel_initializer=init,
                       bias_initializer=zero, name='fc1')
        # Layer 4: Fully Connected. Input = 120. Output = 84.
        self.fc2 = tf.keras.layers.Dense(84, kernel_initializer=init,
                       bias_initializer=zero, name='fc2')
        # Layer 5: Fully Connected. Input = 84. Output = 10.
        self.fc3 = tf.keras.layers.Dense(10, kernel_initializer=init,
                       bias_initializer=zero, name='fc3')

    def call(self, x):
        # Layer 1
        x = tf.nn.relu(self.conv1(x))
        x = self.pool1(x)
        # Layer 2
        x = tf.nn.relu(self.conv2(x))
        x = self.pool2(x)
        # Flatten + FC
        x = self.flat(x)
        x = tf.nn.relu(self.fc1(x))
        x = tf.nn.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits

In [ ]:
model = LeNet()
rate = 0.001

# Build the model so weights are created
model.build(input_shape=(None, 32, 32, 1))
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=rate),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
def evaluate(X_data, y_data):
    _, accuracy = model.evaluate(X_data, y_data, batch_size=BATCH_SIZE, verbose=0)
    return accuracy

In [ ]:
print("Training...")
print()
for i in range(EPOCHS):
    X_train, y_train = shuffle(X_train, y_train)
    model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=1, verbose=0)

    validation_accuracy = evaluate(X_validation, y_validation)
    print("EPOCH {} ...".format(i+1))
    print("Validation Accuracy = {:.3f}".format(validation_accuracy))
    print()

model.save_weights('./lenet')
print("Model saved")

In [ ]:
model.load_weights('./lenet')

test_accuracy = evaluate(X_test, y_test)
print("Test Accuracy = {:.3f}".format(test_accuracy))

In [ ]:
np.set_printoptions(threshold=int(1e6))

model.load_weights('./lenet')

# Extract weights (same shapes as before: kernel is [H,W,Cin,Cout])
conv1_W = model.conv1.kernel.numpy()
conv1_b = model.conv1.bias.numpy()
conv2_W = model.conv2.kernel.numpy()
conv2_b = model.conv2.bias.numpy()
fc1_W   = model.fc1.kernel.numpy()
fc1_b   = model.fc1.bias.numpy()
fc2_W   = model.fc2.kernel.numpy()
fc2_b   = model.fc2.bias.numpy()
fc3_W   = model.fc3.kernel.numpy()
fc3_b   = model.fc3.bias.numpy()

# Compute intermediate outputs for first test image
_inp = X_test[0:1]
_c1  = model.conv1(_inp)
_r1  = tf.nn.relu(_c1)
_p1  = model.pool1(_r1)
_c2  = model.conv2(_p1)
_r2  = tf.nn.relu(_c2)
_p2  = model.pool2(_r2)
_f0  = model.flat(_p2)
_fc1 = tf.nn.relu(model.fc1(_f0))
_fc2 = tf.nn.relu(model.fc2(_fc1))
_logits = model.fc3(_fc2)

fc1_re  = _fc1.numpy()
fc2_re  = _fc2.numpy()

# Also compute logits for first test image
logits_re = _logits.numpy()

####################################################
flog = open("weight1.log", 'w')

weight1 = conv1_W
print("weight1: ", weight1.shape)

for i in range(5):
    for j in range(5):
        for o in range(6):
            if (weight1[i][j][0][o]*np.float64(2**31)).astype(np.int64) < 0 :
                print("-", file = flog, end='')
            print("32'd%d, "%abs((weight1[i][j][0][o]*np.float64(2**31)).astype(np.int64)) , file = flog, end=' ')
        print("", file = flog)
    print("", file = flog)

print("==========================conv1_bias==========================", file = flog)
bias1 = conv1_b
print("", file = flog)
for o in range(6):
    if (bias1[o]*np.float64(2**55)).astype(np.int64) < 0 :
        print("-", file = flog, end='')
    print("56'd%d, "%abs((bias1[o]*np.float64(2**55)).astype(np.int64)), file = flog, end=' ')

flog.close()


####################################################

flog = open("weight2.log", 'w')

weight2 = conv2_W
print("weight2: ", weight2.shape)

for row in range(5):
    for col in range(5):
        for o in range(16):
            for i in range(6):
                if (weight2[row][col][i][o]*np.float64(2**31)).astype(np.int64) < 0 :
                    print("-", file = flog, end='')
                print("32'd%d, "%abs((weight2[row][col][i][o]*np.float64(2**31)).astype(np.int64)) , file = flog, end=' ')
            print("", file = flog)
        print("", file = flog)
    print("", file = flog)

print("", file = flog)

bias2 = conv2_b
print("bias2: ", bias2.shape)
print("", file = flog)
print("", file = flog)
print("==========================conv2_bias==========================", file = flog)
for o in range(16):
    if (bias2[o]*np.float64(2**55)).astype(np.int64) < 0 :
        print("-", file = flog, end='')
    print("56'd%d, "%abs((bias2[o]*np.float64(2**55)).astype(np.int64)), file = flog, end=' ')


flog.close()

####################################################
flog = open("fc1.log", 'w')
weight_fc1 = fc1_W

print("weight_fc1: ", weight_fc1.shape)

for o in range(120):
    for i in range(400):
        if (weight_fc1[i][o]*np.float64(2**31)).astype(np.int64) < 0 :
            print("-", file = flog, end='')
        print("32'd%d, "%abs((weight_fc1[i][o]*np.float64(2**31)).astype(np.int64)) , file = flog, end=' ')
        if i%16==15 :
            print("", file = flog)
    print("", file = flog)
bias_fc1 = fc1_b
print("bias_fc1: ", bias_fc1.shape)
print("==========================bias_fc1==========================", file = flog)
for o in range(120):
    if (bias_fc1[o]*np.float64(2**55)).astype(np.int64) < 0 :
        print("-", file = flog, end='')
    print("56'd%d, "%abs((bias_fc1[o]*np.float64(2**55)).astype(np.int64)) , file = flog, end=' ')
    if o%16==15 :
        print("", file = flog)
print("", file = flog)

print("fc1_re: ", fc1_re.shape)
print("==========================fc1_re==========================", file = flog)

for i in range(120):
    print("%6d, "%(fc1_re[0][i]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
    if i%16==15 :
        print("", file = flog)
print("", file = flog)

flog.close()
####################################################
flog = open("fc2.log", 'w')
weight_fc2 = fc2_W

print("weight_fc2: ", weight_fc2.shape)

for o in range(84):
    for i in range(120):
        if (weight_fc2[i][o]*np.float64(2**31)).astype(np.int64) < 0 :
            print("-", file = flog, end='')
        print("32'd%d, "%abs((weight_fc2[i][o]*np.float64(2**31)).astype(np.int64)) , file = flog, end=' ')
        if i%16==15 :
            print("", file = flog)
    print("", file = flog)
bias_fc2 = fc2_b
print("bias_fc2: ", bias_fc2.shape)
print("==========================bias_fc2==========================", file = flog)
for o in range(84):
    if (bias_fc2[o]*np.float64(2**55)).astype(np.int64) < 0 :
        print("-", file = flog, end='')
    print("56'd%d, "%abs((bias_fc2[o]*np.float64(2**55)).astype(np.int64)) , file = flog, end=' ')
    if o%16==15 :
        print("", file = flog)
print("", file = flog)

print("fc2_re: ", fc2_re.shape)
print("==========================fc2_re==========================", file = flog)

for i in range(84):
    print("%6d, "%(fc2_re[0][i]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
    if i%16==15 :
        print("", file = flog)
print("", file = flog)

flog.close()
####################################################
flog = open("fc3.log", 'w')
weight_fc3 = fc3_W

print("weight_fc3: ", weight_fc3.shape)

for o in range(10):
    for i in range(84):
        if (weight_fc3[i][o]*np.float64(2**31)).astype(np.int64) < 0 :
            print("-", file = flog, end='')
        print("32'd%d, "%abs((weight_fc3[i][o]*np.float64(2**31)).astype(np.int64)) , file = flog, end=' ')
        if i%16==15 :
            print("", file = flog)
    print("", file = flog)
bias_fc3 = fc3_b
print("bias_fc3: ", bias_fc3.shape)
print("==========================bias_fc3==========================", file = flog)
for o in range(10):
    if (bias_fc3[o]*np.float64(2**55)).astype(np.int64) < 0 :
        print("-", file = flog, end='')
    print("56'd%d, "%abs((bias_fc3[o]*np.float64(2**55)).astype(np.int64)) , file = flog, end=' ')
    if o%16==15 :
        print("", file = flog)
print("", file = flog)

fc3_re = logits_re
print("fc3_re: ", fc3_re.shape)
print("==========================fc3_re==========================", file = flog)

for i in range(10):
    print("%6d, "%(fc3_re[0][i]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
    if i%16==15 :
        print("", file = flog)
print("", file = flog)

flog.close()

In [ ]:
# test
np.set_printoptions(threshold=int(1e6))

model.load_weights('./lenet')

# Compute intermediate activations for first test image
_inp = X_test[0:1]
conv1_out  = model.conv1(_inp).numpy()                          # raw conv1
relu1_out  = tf.nn.relu(model.conv1(_inp)).numpy()              # after relu
pool1_out  = model.pool1(tf.nn.relu(model.conv1(_inp))).numpy() # after pool

_p1 = model.pool1(tf.nn.relu(model.conv1(_inp)))
conv2_out  = (model.conv2(_p1)).numpy()
pool2_out  = model.pool2(tf.nn.relu(model.conv2(_p1))).numpy()
fc0_out    = model.flat(model.pool2(tf.nn.relu(model.conv2(_p1)))).numpy()

####################################################
flog = open("conv1.log", 'w')

conv_re  = conv1_out
relu1_re = relu1_out
pool1_re = pool1_out
print("conv1: ", conv_re.shape)
print("relu1: ", relu1_re.shape)
print("pool1_re: ", pool1_re.shape)


print("conv1: ", conv_re.shape, file = flog)
for o in range(6):
    for i in range(28):
        for j in range(28):
            print(conv_re[0][i][j][o], file = flog, end=' ')
        print("", file = flog)
    print("", file = flog)


print("", file = flog)
for i in range(28):
    for j in range(28):
        for o in range(6):
            print("%6d, "%(conv_re[0][i][j][o]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
        print("", file = flog)
    print("", file = flog)

print("==========================pool1_re==========================", file = flog)

print("", file = flog)
for i in range(14):
    for j in range(14):
        for o in range(6):
            print("%6d, "%(pool1_re[0][i][j][o]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
        print("", file = flog)
    print("", file = flog)
flog.close()

####################################################

flog = open("conv2.log", 'w')

conv2_re = conv2_out
print("conv2: ", conv2_re.shape)

print("", file = flog)
for i in range(10):
    for j in range(10):
        for o in range(16):
            print("%d, "%(conv2_re[0][i][j][o]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
        print("", file = flog)
    print("", file = flog)

pool2_re = pool2_out
print("==========================pool2_re==========================", file = flog)
print("pool2_re: ", pool2_re.shape)
print("", file = flog)
for i in range(5):
    for j in range(5):
        for o in range(16):
            print("%6d, "%(pool2_re[0][i][j][o]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
        print("", file = flog)
    print("", file = flog)


fc0_re = fc0_out
print("fc0_re: ", fc0_re.shape)
print("==========================fc0_re==========================", file = flog)

for i in range(400):
    print("%6d, "%(fc0_re[0][i]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
    if i%16==15 :
        print("", file = flog)
flog.close()


####################################################

flog = open("test_digits.log", 'w')
print("X_test: ", X_test.shape)
for n in range(100):
    for i in range(32):
        for j in range(32):
            print("%4d, "%(X_test[n][i][j][0]*255).astype(int), file = flog, end='')
        print("", file = flog)
    print("", file = flog)
    print("", file = flog)

fc3_re = model(_inp:=X_test[0:100]).numpy()
print("fc3_re: ", fc3_re.shape)
print("==========================fc3_re==========================", file = flog)

for n in range(100):
    for i in range(10):
        print("%6d, "%(fc3_re[n][i]*np.float64(2**24)).astype(np.int64), file = flog, end=' ')
    print("", file = flog)


print("==========================digits==========================", file = flog)
print("y_test: ", y_test.shape)
for n in range(100):
    print("%d, %d"% (n, y_test[n]), file = flog, end=' ')
    print("", file = flog)

print("", file = flog)

flog.close()